# AIND Ephys Dispatch

Build an `AINDEPhysDispatchScanConfig`, expand it with `GridScanGenerationTask`, and run the `AINDEPhysDispatchTask` for each coordinate. The task itself clones [aind-ephys-job-dispatch](https://github.com/AllenNeuralDynamics/aind-ephys-job-dispatch) on first use, invokes its `code/run_capsule.py` with the flags this single config produces, and copies the resulting `job_*.json` files into the single config's `coordinate_output_root`.

This notebook targets the toy SpikeInterface recording from `00_generate_toy_recording.ipynb` (saved at `./output/00_toy_example_recording`). It is a `BinaryFolderRecording`, so we use the `spikeinterface` input mode with `reader_type='binaryfolder'`.

## Imports

In [1]:
import json
import shutil
from pathlib import Path

import obi_one as obi
from obi_one.scientific.tasks.spike_sorting.dispatch.blocks import (
    DispatchBasic,
    DispatchDataDependent,
    DispatchDebug,
    SpikeInterfaceInfo,
)

## Build the scan config

Any field set to a list becomes a dimension of the parameter scan. Below we sweep `min_recording_duration` (2 values) and `debug_mode` (2 values), giving a 2x2 = 4 coordinate scan.

The recording is loaded by SpikeInterface via the `binaryfolder` extractor, pointing at the saved toy recording with an absolute path.

In [2]:
recording_folder = (Path.cwd() / "output/00_toy_example_recording").resolve()
assert recording_folder.exists(), (
    f"{recording_folder} not found. Run 00_generate_toy_recording.ipynb first."
)

scan_config = obi.AINDEPhysDispatchScanConfig(
    dispatch_basic=DispatchBasic(
        split_segments=True,
        split_groups=True,
        skip_timestamps_check=False,
        min_recording_duration=1.0,
    ),
    dispatch_data_dependent=DispatchDataDependent(
        input_format="spikeinterface",
        multi_session_data=False,
        spikeinterface_info=SpikeInterfaceInfo(
            reader_type="binaryfolder",
            reader_kwargs={"folder_path": str(recording_folder)},
        ),
    ),
    dispatch_debug=DispatchDebug(
        debug_mode=False,
        debug_duration=10.0,
    ),
)

## Generate the grid scan and run the dispatch task for each coordinate

`run_tasks_for_generated_scan` calls `AINDEPhysDispatchTask.execute()` once per single config. Each call invokes `code/run_capsule.py` and writes the resulting `job_*.json` into that single config's `coordinate_output_root`.

In [3]:
grid_scan = obi.GridScanGenerationTask(
    form=scan_config,
    output_root="../../../obi-output/01_dispatch_results/grid_scan",
)
grid_scan.execute()

obi.run_tasks_for_generated_scan(grid_scan)

[2026-04-29 16:28:20,944] INFO: python -u code/run_capsule.py --input spikeinterface --min-recording-duration 1.0 --spikeinterface-info '{"reader_type": "binaryfolder", "reader_kwargs": {"folder_path": "/Users/james/Documents/obi/code/obi-main/obi-one/examples/K_extracellular/output/00_toy_example_recording"}}'


[2026-04-29 16:28:21,280] INFO: Running job dispatcher with the following parameters:
	SPLIT SEGMENTS: True
	SPLIT GROUPS: True
	DEBUG: False
	DEBUG DURATION: 30.0
	SKIP TIMESTAMPS CHECK: False
	MULTI SESSION: False
	INPUT: spikeinterface
	MIN_RECORDING_DURATION: 1.0
	SPIKEINTERFACE_INFO: {"reader_type": "binaryfolder", "reader_kwargs": {"folder_path": "/Users/james/Documents/obi/code/obi-main/obi-one/examples/K_extracellular/output/00_toy_example_recording"}}
Parsing spikeinterface input folder
	Stream name: None
Recording to be processed in parallel:
Relative to: ../data
	block0_None_recording1
		Duration: 10.0 s - Num. channels: 70
Generated 1 job config files



[PosixPath('../../../obi-output/01_dispatch_results/grid_scan/AINDEPhysDispatchSingleConfig')]

## Copy the generated job files next to the notebook

Each coordinate has its own `coordinate_output_root` outside this folder (under `obi-output/`). For convenience we copy the `job_*.json` files into `output/01_dispatch_results/<idx>/` so later notebooks can find them at a stable path.

In [4]:
local_results_dir = Path.cwd() / "output/01_dispatch_results"
if local_results_dir.exists():
    shutil.rmtree(local_results_dir)
local_results_dir.mkdir(parents=True, exist_ok=True)

for single_config in grid_scan.single_configs:
    coord_dir = Path(single_config.coordinate_output_root)
    job_files = sorted(coord_dir.glob("job_*.json"))
    dest = local_results_dir / str(single_config.idx)
    dest.mkdir(parents=True, exist_ok=True)
    for jf in job_files:
        shutil.copy2(jf, dest / jf.name)

# Also keep a top-level copy of the first coordinate's job_0.json so downstream
# notebooks have a stable `output/01_dispatch_results/job_0.json` to read.
first_job = local_results_dir / "0" / "job_0.json"
if first_job.exists():
    shutil.copy2(first_job, local_results_dir / "job_0.json")

print("Copied job files to:", local_results_dir)

Copied job files to: /Users/james/Documents/obi/code/obi-main/obi-one/examples/K_extracellular/output/01_dispatch_results


## Inspect the generated job JSON

The dispatcher writes one `job_*.json` per detected recording. We've copied them next to this notebook at `./output/01_dispatch_results/`. Each describes a serialized SpikeInterface recording the downstream preprocessing/sorting capsules consume.

In [5]:
job_files = sorted(local_results_dir.rglob("job_*.json"))
print(f"Found {len(job_files)} job file(s) under {local_results_dir.name}/:")
for jf in job_files:
    print(" ", jf.relative_to(local_results_dir))

if (local_results_dir / "job_0.json").exists():
    job = json.loads((local_results_dir / "job_0.json").read_text())
    print()
    print("Top-level keys:", list(job.keys()))
    print("session_name:  ", job.get("session_name"))
    print("recording_name:", job.get("recording_name"))

Found 2 job file(s) under 01_dispatch_results/:
  0/job_0.json
  job_0.json

Top-level keys: ['session_name', 'recording_name', 'recording_dict', 'skip_times', 'duration', 'input_folder', 'debug', 'multi_input']
session_name:   session0
recording_name: block0_None_recording1
